# 01 — Analyse exploratoire des données (EDA)

Jeu synthétique `data/synthetic/dataset.csv` (3 000 profils, seed 42).
Méthode de génération, hypothèses et biais : voir `data/synthetic/DONNEES-SYNTHETIQUES.md`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/synthetic/dataset.csv')
print(df.shape)
df.head()

(3000, 12)


,id,serie_bac,note_maths,note_sciences,note_langues,note_eco,matieres_preferees,competences,interets,environnement,metiers_vises,filiere
0,syn-00000,S,4,4,3,4,Physique-Chimie|Histoire-Geographie|Arts,Electronique / bricolage technique|Analyse de ...,Art / design / audiovisuel|Sciences|Droit / ju...,Terrain / exterieur,Creation / design,EMII
1,syn-00001,C,4,4,3,3,Arts|Economie / Gestion,Creativite / design|Vente / negociation|Organi...,Finance / comptabilite|Communication / medias|...,Atelier / usine,Commerce / relation client,CAA
2,syn-00002,S,4,4,4,3,Physique-Chimie|Francais / Litterature|Sport,Creativite / design|Redaction / communication|...,BTP / construction|Art / design / audiovisuel|...,Terrain / exterieur,Commerce / relation client,IMTICIA
3,syn-00003,A1,3,3,4,2,Economie / Gestion|Informatique / Technologie|SVT,Organisation / gestion de projet|Travail en eq...,Sciences|Art / design / audiovisuel|Entreprene...,Mixte,Entrepreneur / independant|Creation / design,CAA
4,syn-00004,D,4,5,5,2,Economie / Gestion|Sport|Francais / Litterature,Redaction / communication|Vente / negociation,Entrepreneuriat / business|Communication / med...,Mixte,Recherche / enseignement,DTJA


In [ ]:
# Équilibre des classes : CAA domine (~14 %), IAA est rare (~1,6 %).
ax = df['filiere'].value_counts().sort_values().plot.barh(figsize=(7, 6), color='#1e6b45')
ax.set_title('Répartition des filières recommandées')
ax.set_xlabel('Nombre de profils')
plt.tight_layout(); plt.show()

In [ ]:
# Notes moyennes par filière : vérifie les corrélations attendues
# (maths élevées pour ISAIA, langues pour TEH/DTJA...).
notes = df.groupby('filiere')[['note_maths', 'note_sciences', 'note_langues', 'note_eco']].mean()
notes.round(2).sort_values('note_maths', ascending=False)

In [ ]:
# Intérêt dominant par filière (top 3) : cohérence profil <-> étiquette.
expl = df.assign(interet=df['interets'].str.split('|')).explode('interet')
top = (expl.groupby(['filiere', 'interet']).size().rename('n').reset_index()
         .sort_values(['filiere', 'n'], ascending=[True, False])
         .groupby('filiere').head(3))
top.pivot_table(index='filiere', columns='interet', values='n', fill_value=0)

In [ ]:
# Distribution des séries de bac et des environnements de travail.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df['serie_bac'].value_counts().plot.bar(ax=axes[0], color='#1e6b45', title='Séries de bac')
df['environnement'].value_counts().plot.bar(ax=axes[1], color='#a85d1c', title='Environnements')
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

## Lectures

- **Déséquilibre de classes** réel mais gérable → `class_weight='balanced'` à l'entraînement.
- Les corrélations notes ↔ filières correspondent aux règles de génération documentées : le modèle
  apprendra principalement ces règles (limite nommée par le sujet), d'où la validation sur l'enquête réelle.
- Aucune valeur manquante ni aberrante par construction ; les réponses d'enquête, elles, devront être nettoyées.